<a href="https://colab.research.google.com/github/J7NU/gemma4/blob/main/multiagent_colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# ============================================================
#  시즌제 멀티에이전트 시스템 — 수정판 (v2)
#  LangGraph (오케스트레이션) + CrewAI (에이전트 협업)
#  Google Colab 실행용 단일 파일
#
#  [v1 대비 수정 사항 — 7개 버그 / 4개 다이어그램 오류 수정]
#  1. crew_setup 노드 추가          → 다이어그램 ③과 코드 graph 동기화
#  2. 수렴 감지 로직 구현           → Jaccard 유사도로 이전/현재 출력 비교
#  3. Mock CP 실패 시뮬레이션        → 첫 시도 fail → 재진입 경로 테스트 가능
#  4. CP3 거절 피드백 → idea 통합   → node_process_input에서 user_feedback 반영
#  5. active_modules 처리 완성      → create_core_crew에 모듈 에이전트 동적 추가
#  6. AgentState 필드 추가          → crew_config, level1_prev_output, convergence_reached
#  7. 카운터 표시 수정              → 시도 횟수 분모를 MAX_RETRIES로 정정
#
#  [실행 순서]
#  1. CELL 1 설치 명령어 주석 해제 후 실행
#  2. 런타임 재시작 (중요)
#  3. CELL 2에서 API 키 설정 또는 MOCK_MODE = True 유지
#  4. 런타임 > 모두 실행 (Ctrl+F9)
# ============================================================

In [ ]:
# ===========================================================
#  ⬇ CELL 1 — 패키지 설치 (처음 실행 시 이 줄들 주석 해제)
# ===========================================================
# !pip install -q \
#     langgraph==0.2.53 \
#     langchain==0.3.14 \
#     langchain-core==0.3.29 \
#     langchain-anthropic==0.3.3 \
#     crewai==0.86.0 \
#     crewai-tools==0.17.0
#
# 설치 완료 후 런타임 재시작:
# import os; os.kill(os.getpid(), 9)


# ===========================================================

In [ ]:
#  ⬇ CELL 2 — 임포트 & 전역 설정
# ===========================================================

import os
import json
import time
from typing import TypedDict, List, Dict, Any, Optional

# ── LangGraph ──────────────────────────────────────────────
from langgraph.graph import StateGraph, START, END
from langgraph.checkpoint.memory import MemorySaver

# ── LangChain (체크포인트 노드용 Claude 호출) ────────────────
from langchain_core.messages import HumanMessage

# ── CrewAI (Level 1·2 에이전트 협업) ────────────────────────
from crewai import Agent, Task, Crew, Process
from crewai.llm import LLM


# ----------------------------------------------------------
#  API 키 설정
#  Colab Secrets(왼쪽 🔑 아이콘)에 ANTHROPIC_API_KEY 등록 권장
# ----------------------------------------------------------
try:
    from google.colab import userdata
    os.environ["ANTHROPIC_API_KEY"] = userdata.get("ANTHROPIC_API_KEY")
    print("✅ Colab Secrets에서 API 키 로드 완료")
except Exception:
    # os.environ["ANTHROPIC_API_KEY"] = "sk-ant-..."
    print("⚠️  ANTHROPIC_API_KEY 미설정 → MOCK_MODE=True 권장")

# ----------------------------------------------------------
#  핵심 파라미터
# ----------------------------------------------------------
MOCK_MODE = True        # False = 실제 API 호출 (API 키 필요)
MAX_RETRIES = 3         # 각 레벨 최대 재진입 횟수 (초과 시 강제 진행)
CONVERGENCE_SIMILARITY = 0.80  # [수정 2] Jaccard 유사도 임계값: 0.8 이상 = 수렴




In [ ]:
# ===========================================================
#  ⬇ CELL 3 — 규칙 정의 (MASTER · PROJECT)
# ===========================================================

MASTER_RULES = f"""
[MASTER_CLAUDE.md — 전역 규칙 (모든 시즌·프로젝트 공통)]

완결성  : 모든 산출물은 실행 가능한 수준으로 완성되어야 한다.
일관성  : 에이전트 간 결과물이 상호 모순되지 않아야 한다.
규칙준수: 각 에이전트는 자신의 역할 범위를 벗어나지 않는다.
추적가능: 모든 결정과 변경 사항은 세션 로그에 기록된다.

[수렴 기준]
- 이전 라운드 대비 Jaccard 유사도 {CONVERGENCE_SIMILARITY * 100:.0f}% 이상이면 수렴으로 간주
- 최대 재진입 횟수: {MAX_RETRIES}회 초과 시 강제 다음 단계 진행
"""


def build_project_rules(project_name: str, output_format: str,
                         constraints: str, experience: str = "없음") -> str:
    """
    PROJECT_CLAUDE.md 를 동적으로 생성한다.
    [슬롯]은 인풋 시점과 이전 시즌 경험으로 채워진다.
    """
    return f"""
[PROJECT_CLAUDE.md — 프로젝트 규칙 (전역 규칙 상속 + 오버라이드)]

프로젝트명  : {project_name}
출력 형식   : {output_format}
핵심 제약   : {constraints}
경험 기준   : {experience}  ← 이전 시즌 아카이브에서 자동 누적
"""




In [ ]:
# ===========================================================
#  ⬇ CELL 4 — LangGraph 상태(AgentState) 정의
#  그래프의 모든 노드가 읽고 쓰는 공유 메모리 구조체
#
#  [수정 6] 신규 필드 3개 추가:
#    · crew_config        : crew_setup 노드에서 결정한 에이전트 구성
#    · level1_prev_output : 수렴 감지용 이전 Level 1 결합 출력
#    · convergence_reached: 수렴 도달 여부 플래그
# ===========================================================

class AgentState(TypedDict):
    # ── 규칙 ────────────────────────────────────────────────
    master_rules:  str
    project_rules: str

    # ── 인풋 ────────────────────────────────────────────────
    idea:           str             # 사용자 아이디어 (CP3 거절 시 피드백 통합)
    project_name:   str
    output_format:  str
    constraints:    str
    active_modules: List[str]       # ["designer", "marketer", "bm"] 등

    # ── [수정 6] 에이전트 구성 (crew_setup 노드 결과) ─────────
    crew_config: Dict[str, Any]     # {core: [...], modules: {name: {role, goal}}}

    # ── Level 1 결과 ────────────────────────────────────────
    level1_strategist:  str
    level1_executor:    str
    level1_critic:      str
    level1_modules:     Dict[str, str]  # {모듈명: 결과}
    level1_retry_count: int
    # [수정 6] 수렴 감지용 이전 출력 저장
    level1_prev_output:   str       # 이전 라운드 Level 1 결합 출력
    convergence_reached:  bool      # 수렴 도달 여부

    # ── Level 2 결과 ────────────────────────────────────────
    level2_integrated:  str
    level2_conflicts:   str
    level2_retry_count: int

    # ── 체크포인트 결과 ────────────────────────────────────────
    cp1_result:   str   # "pass" | "fail"
    cp1_feedback: str
    cp2_result:   str
    cp2_feedback: str

    # ── 사용자 최종 승인 ────────────────────────────────────────
    final_proposal: str
    user_decision:  str   # "approved" | "rejected"
    user_feedback:  str   # 거절 시 피드백 → 다음 idea에 통합

    # ── 실행 & 로그 ────────────────────────────────────────────
    execution_result: str
    session_log:      List[str]




In [ ]:
# ===========================================================
#  ⬇ CELL 5 — [수정 2] 수렴 감지 유틸리티
# ===========================================================

def jaccard_similarity(text1: str, text2: str) -> float:
    """
    두 텍스트 간 Jaccard 유사도를 계산한다.

    Jaccard = |교집합| / |합집합|  (토큰 집합 기준)
    반환값: 0.0 (완전 다름) ~ 1.0 (완전 동일)

    수렴 판정 기준: CONVERGENCE_SIMILARITY (기본 0.8) 이상
    → 이전 라운드와 80% 이상 유사하면 더 이상 개선되지 않는다고 판단
    """
    if not text1 or not text2:
        return 0.0
    tokens1 = set(text1.lower().split())
    tokens2 = set(text2.lower().split())
    if not tokens1 or not tokens2:
        return 0.0
    intersection = len(tokens1 & tokens2)
    union        = len(tokens1 | tokens2)
    return intersection / union if union > 0 else 0.0


def combine_level1_outputs(state: AgentState) -> str:
    """Level 1 전체 출력을 단일 문자열로 결합 (수렴 비교용)"""
    parts = [
        state.get("level1_strategist", ""),
        state.get("level1_executor", ""),
        state.get("level1_critic", ""),
        " ".join(state.get("level1_modules", {}).values()),
    ]
    return " ".join(parts)




In [ ]:
# ===========================================================
#  ⬇ CELL 6 — Mock LLM 응답 (API 없이 흐름 테스트용)
#  [수정 3] Mock CP: 첫 시도(retry=0) → fail (재진입 루프 테스트 가능)
# ===========================================================

def mock_strategist_response(idea: str, retry: int = 0) -> str:
    suffix = f" [재시도 {retry}회차 — 이전 비판 반영]" if retry > 0 else ""
    return f"""
[전략가 — 방향·기획]{suffix}
핵심 목표 : "{idea[:40]}..." → MVP 형태로 5주 내 출시
우선순위  : 1) 핵심 기능 정의 → 2) 경쟁사 분석 → 3) 사용자 검증 → 4) 수익화
성공 지표 : 초기 100명 사용자, NPS 40+, 3개월 내 유료 전환 10%
리스크    : 시장 포화, 개발 일정 초과, 초기 수익화 어려움
차별화    : {"재시도로 경쟁사 포지셔닝 명시됨" if retry > 0 else "경쟁사 분석 추가 필요"}
"""


def mock_executor_response(idea: str, retry: int = 0) -> str:
    weeks = "6주" if retry > 0 else "4주 (비현실적 — 재산정 필요)"
    return f"""
[실행가 — 구체적 구현] [재시도 {retry}회차 반영]
기술 스택 : Next.js 14 + FastAPI + PostgreSQL + Redis
주차별 계획:
  Week 1~2: DB 설계 · API 스켈레톤 · 사용자 인터뷰 병행
  Week 3~4: 핵심 기능 구현 · 온보딩 UX 우선 적용
  Week 5  : UI 완성 · 테스트 · 성능 최적화
  Week 6  : 배포 · 모니터링 설정 (조정: {weeks})
예상 비용 : 서버 월 7만원 (Vercel Pro + Supabase)
"""


def mock_critic_response(retry: int = 0,
                          converging: bool = False) -> str:
    if converging:
        return (
            "[비판가] 이전 라운드 지적 사항 모두 반영 확인.\n"
            "· 경쟁사 포지셔닝 추가 ✓\n"
            "· 일정 현실화(5~6주) ✓\n"
            "· 수익화 시점 명확화 ✓\n"
            "추가 맹점 없음 → 수렴 판정"
        )
    return f"""
[비판가 — 맹점·리스크 발굴] [라운드 {retry+1}]
맹점 1 : 경쟁사 분석 누락 → 차별화 전략 필요
맹점 2 : {retry+1}인 개발 기준 {4+retry}주 일정 비현실적 → {5+retry}주 권장
맹점 3 : 수익화 시점 불명확 → 3개월 후 프리미엄 전환 가정 필요
권고   : 전략가는 경쟁사 포지셔닝, 실행가는 일정 재산정 필요
"""


def mock_cross_feedback(l1_results: Dict, retry: int = 0) -> tuple[str, str]:
    """Mock Level 2 크로스 피드백 → (통합 결과, 충돌 목록)"""
    integrated = f"""
[Level 2 크로스 피드백 통합안] [재시도 {retry}회차]
충돌 해결:
  · 일정 충돌 → 전략가·실행가 합의: 5주(+1주 버퍼)로 조정 ✓
  · 수익화 시점 → BM 에이전트 제안 반영: 출시 3개월 후 유료 전환 ✓
통합 개선:
  · 전략 + 실행 연계: Week 1~2에 사용자 인터뷰 병행 반영
  · 디자이너: 온보딩 UX 우선 적용 반영
  · 마케터: 출시 전 대기자 명단 확보 전략 추가
최종 통합: 모든 에이전트 의견 수렴 완료
"""
    conflicts = "· 일정 충돌 (전략가 5주 vs 실행가 6주)\n· 수익화 시점 불명확"
    return integrated, conflicts


# [수정 3] Mock CP: 첫 시도 fail → 재진입 루프 테스트 가능
def mock_claude_cp_review(level: int, retry: int = 0) -> tuple[str, str]:
    """
    수정 포인트: retry=0(첫 시도)은 fail을 반환해
    CP → Level 재진입 경로가 실제로 동작함을 검증한다.
    retry>=1 이면 pass를 반환해 다음 단계로 진행한다.
    """
    if retry == 0:
        # 첫 시도: 비판가 지적 미반영으로 실패 처리
        return "fail", (
            f"[CP{level}] 초기 검토 실패: "
            f"비판가 지적 사항(경쟁사 분석, 일정 현실화)이 전략·실행에 미반영. "
            f"Level {level} 재시도 필요."
        )
    # 재시도: 개선 확인 후 통과
    return "pass", (
        f"[CP{level}] 재시도 {retry}회차 통과: "
        f"완결성·일관성·규칙 준수 확인. 비판가 지적 반영 확인. 통과."
    )


def mock_final_proposal(state: AgentState) -> str:
    """Mock 최종 통합 제안서"""
    modules_str = ", ".join(state.get("active_modules", [])) or "없음"
    return f"""
╔══════════════════════════════════════════════════════════╗
║                 최종 통합 제안서                          ║
╠══════════════════════════════════════════════════════════╣
║  프로젝트 : {state['project_name']}
║  출력형식 : {state['output_format']}
║  활성모듈 : {modules_str}
╠══════════════════════════════════════════════════════════╣
║  [전략 요약]
║  {state.get('level1_strategist', '')[:120].strip().replace(chr(10), ' | ')}
╠══════════════════════════════════════════════════════════╣
║  [실행 요약]
║  {state.get('level1_executor', '')[:120].strip().replace(chr(10), ' | ')}
╠══════════════════════════════════════════════════════════╣
║  [통합 크로스피드백 요약]
║  {state.get('level2_integrated', '')[:120].strip().replace(chr(10), ' | ')}
╚══════════════════════════════════════════════════════════╝
    """.strip()




In [ ]:
# ===========================================================
#  ⬇ CELL 7 — CrewAI 에이전트 & 태스크 빌더
#  [수정 5] crew_config 파라미터 추가 → active_modules 에이전트 동적 생성
# ===========================================================

def build_llm(model: str = "claude-sonnet-4-20250514") -> Optional[LLM]:
    """
    CrewAI 에이전트용 LLM 생성.

    [실제 환경에서 Gemma(저비용 LLM) 사용법]
    Level 1·2는 비용을 줄이기 위해 Gemma 등 소형 모델 사용 권장:
      # Groq 무료 API
      return LLM(model="groq/gemma2-9b-it",
                 api_key=os.environ["GROQ_API_KEY"])

      # Ollama 로컬 실행 (무료, 고성능 GPU 권장)
      return LLM(model="ollama/gemma2:9b",
                 base_url="http://localhost:11434")

    Claude(고비용)는 체크포인트 3회에만 사용해 비용을 최소화한다.
    """
    if MOCK_MODE:
        return None  # Mock 모드에서는 LLM 생성 스킵
    return LLM(
        model=f"anthropic/{model}",
        api_key=os.environ.get("ANTHROPIC_API_KEY", ""),
    )


# [수정 5] crew_config를 받아 코어 + 선택 모듈 에이전트를 동적으로 생성
def create_core_crew(idea: str, rules: str,
                     crew_config: Dict[str, Any],
                     retry: int = 0) -> Crew:
    """
    Level 1 코어 크루 생성.

    crew_config 구조:
      {
        "core":    ["strategist", "executor", "critic"],
        "modules": {
          "designer": {"role": "UI/UX 디자이너", "goal": "..."},
          "marketer":  {"role": "마케터",         "goal": "..."},
        }
      }

    Process.SEQUENTIAL: 전략가 → 실행가 → 비판가 → [모듈들] 순서 실행
    각 에이전트 출력이 다음 에이전트의 context로 전달됨.
    """
    llm = build_llm()

    # ── 코어: 전략가 ─────────────────────────────────────────
    strategist = Agent(
        role="전략가",
        goal="아이디어를 실행 가능한 전략과 우선순위로 변환한다",
        backstory=f"""당신은 스타트업 전략 전문가입니다.
        아이디어를 시장 맥락에서 분석하고 명확한 목표·우선순위를 설정합니다.
        이번은 {retry}번째 재시도입니다. 이전 비판가 지적을 반드시 반영하세요.
        규칙: {rules[:200]}""",
        verbose=True, llm=llm,
    )

    # ── 코어: 실행가 ─────────────────────────────────────────
    executor = Agent(
        role="실행가",
        goal="전략을 구체적인 기술 구현 계획으로 변환한다",
        backstory=f"""당신은 풀스택 개발자이자 프로젝트 매니저입니다.
        전략 방향을 실제 코드와 현실적인 일정으로 구체화하는 것이 역할입니다.
        이번은 {retry}번째 재시도입니다. 일정 현실성에 특히 주의하세요.""",
        verbose=True, llm=llm,
    )

    # ── 코어: 비판가 ─────────────────────────────────────────
    critic = Agent(
        role="비판가",
        goal="전략과 실행 계획의 맹점, 리스크, 비일관성을 발굴한다",
        backstory="""당신은 냉철한 비즈니스 컨설턴트입니다.
        어떤 계획에서도 빠진 부분, 과도한 낙관, 숨겨진 리스크를 발굴합니다.
        건설적이되 타협 없는 비판을 제시하세요.""",
        verbose=True, llm=llm,
    )

    # ── 태스크 정의 ──────────────────────────────────────────
    task_strategist = Task(
        description=f"""
아이디어: {idea}
적용 규칙: {rules[:300]}
현재 재시도 횟수: {retry}회 (0이면 첫 시도)

전략을 수립하세요:
- 핵심 목표 (1~2문장)
- 우선순위 목록 (번호 순)
- 성공 지표 (정량 기준)
- 예상 리스크와 대응
- 경쟁사 대비 차별화 포인트 (재시도 시 반드시 포함)
        """,
        expected_output="구체적이고 실행 가능한 전략 문서",
        agent=strategist,
    )

    task_executor = Task(
        description=f"""
전략가 결과를 바탕으로 구체적인 실행 계획을 수립하세요.
아이디어: {idea}
재시도 횟수: {retry}회

필수 포함:
- 기술 스택 선택 및 이유
- 주차별 구현 계획 (현실적 일정, 재시도 시 일정 재산정 필수)
- 예상 비용
- 필요 리소스
        """,
        expected_output="주차별 실행 계획과 기술 스펙",
        agent=executor,
        context=[task_strategist],
    )

    task_critic = Task(
        description=f"""
전략가와 실행가의 결과를 비판적으로 검토하세요.
규칙: {rules[:200]}

검토 항목:
1. 전략의 현실성 및 시장 분석 완결성
2. 실행 계획의 타당성 및 일정 현실성
3. 두 결과 간 불일치 또는 모순
4. 누락된 핵심 요소 (경쟁사 분석, 수익화 시점 등)

출력: 맹점 목록 + 각 에이전트별 개선 권고
        """,
        expected_output="맹점 목록과 구체적 개선 방안",
        agent=critic,
        context=[task_strategist, task_executor],
    )

    agents = [strategist, executor, critic]
    tasks  = [task_strategist, task_executor, task_critic]

    # ── [수정 5] 선택 모듈 에이전트 동적 추가 ─────────────────
    # crew_config["modules"]에 등록된 모듈만 생성
    for mod_name, mod_info in crew_config.get("modules", {}).items():
        mod_agent = Agent(
            role=mod_info["role"],
            goal=mod_info["goal"],
            backstory=f"""당신은 {mod_info['role']}입니다.
            {mod_info['goal']}
            전략가와 실행가의 결과를 참고해 당신의 전문 분야 관점을 추가하세요.""",
            verbose=True,
            llm=llm,
        )

        mod_task = Task(
            description=f"""
아이디어: {idea}
당신의 역할: {mod_info['role']}

전략가 · 실행가 결과를 참고해 당신의 전문 관점에서 핵심 제안을 작성하세요.
목표: {mod_info['goal']}
출력: {mod_info['role']} 관점의 구체적 액션 아이템 3~5개
            """,
            expected_output=f"{mod_info['role']} 관점 핵심 제안",
            agent=mod_agent,
            context=[task_strategist, task_executor],
        )

        agents.append(mod_agent)
        tasks.append(mod_task)

    return Crew(
        agents=agents,
        tasks=tasks,
        process=Process.sequential,
        verbose=True,
    )


def create_cross_feedback_crew(l1_results: Dict,
                                rules: str,
                                crew_config: Dict[str, Any]) -> Crew:
    """
    Level 2 크로스 피드백 크루.
    [수정 5] crew_config의 모듈 출력도 통합 조율자에게 전달.
    """
    llm = build_llm()

    # 모듈 출력 포매팅 (crew_config 기반)
    modules_summary = ""
    for mod_name, mod_info in crew_config.get("modules", {}).items():
        mod_output = l1_results.get("modules", {}).get(mod_name, "출력 없음")
        modules_summary += f"\n[{mod_info['role']} 결과]\n{mod_output}\n"

    integrator = Agent(
        role="통합 조율자",
        goal="모든 에이전트 관점을 조율하고 최적 통합안을 도출한다",
        backstory="""당신은 다양한 관점을 통합하는 퍼실리테이터입니다.
        전략·실행·비판·전문 모듈의 관점 충돌을 파악하고,
        모든 핵심 요소가 반영된 통합안을 만드는 것이 역할입니다.""",
        verbose=True,
        llm=llm,
    )

    cross_task = Task(
        description=f"""
아래 Level 1 에이전트 결과를 크로스 검토하고 통합하세요.

[전략가 결과]
{l1_results.get('strategist', '없음')}

[실행가 결과]
{l1_results.get('executor', '없음')}

[비판가 결과]
{l1_results.get('critic', '없음')}

[선택 모듈 결과]
{modules_summary or '선택 모듈 없음'}

적용 규칙: {rules[:300]}

수행 작업:
1. 에이전트 간 충돌 지점 식별 및 목록화
2. 충돌 해결 방안 제시 (양측 균형)
3. 비판가 지적이 전략·실행에 반영되었는지 검증
4. 모듈 에이전트 제안의 통합 여부 확인
5. 통합된 최종 방향 제시
        """,
        expected_output="충돌 목록 + 통합 해결안 + 최종 방향",
        agent=integrator,
    )

    return Crew(
        agents=[integrator],
        tasks=[cross_task],
        process=Process.sequential,
        verbose=True,
    )




In [ ]:
# ===========================================================
#  ⬇ CELL 8 — LangGraph 노드 함수 정의
# ===========================================================

def log(state: AgentState, message: str) -> List[str]:
    """타임스탬프와 함께 세션 로그에 추가"""
    entry = f"[{time.strftime('%H:%M:%S')}] {message}"
    print(f"  📋 {entry}")
    return state.get("session_log", []) + [entry]


# ── 노드 1: 규칙 로드 ──────────────────────────────────────
def node_load_rules(state: AgentState) -> AgentState:
    """
    MASTER_CLAUDE.md 와 PROJECT_CLAUDE.md 를 로드해 State에 저장.
    실제 환경에서는 파일시스템 또는 DB에서 읽어옴.
    """
    print("\n" + "="*60)
    print("① 규칙 계층 로드")
    print("="*60)

    project_rules = build_project_rules(
        project_name  = state.get("project_name", "미정"),
        output_format = state.get("output_format", "웹앱"),
        constraints   = state.get("constraints", "없음"),
        experience    = "없음 (첫 시즌)",
    )

    return {
        **state,
        "master_rules":  MASTER_RULES,
        "project_rules": project_rules,
        "session_log":   log(state, "규칙 로드 완료 (MASTER + PROJECT)"),
    }


# ── 노드 2: 인풋 처리 ──────────────────────────────────────
# [수정 4] CP3 거절 시 user_feedback → idea 통합
def node_process_input(state: AgentState) -> AgentState:
    """
    사용자 인풋을 검증하고 활성 모듈을 확정한다.
    CP3 거절 후 재진입 시에는 user_feedback을 idea에 통합해
    다음 시즌이 이전 피드백을 반영한 상태에서 시작되도록 한다.
    """
    print("\n" + "="*60)
    print("② 인풋 처리")
    print("="*60)

    user_feedback = state.get("user_feedback", "")
    original_idea = state.get("idea", "")

    # [수정 4] CP3 거절 피드백 → idea에 통합 후 클리어
    if user_feedback and user_feedback not in ("", "사용자가 세션을 종료했습니다."):
        updated_idea = (
            f"{original_idea}\n\n"
            f"[사용자 수정 요청 — 이전 거절 피드백]\n{user_feedback}"
        )
        print(f"  ♻️  이전 거절 피드백 통합:\n     {user_feedback[:80]}...")
    else:
        updated_idea = original_idea

    print(f"  💡 아이디어  : {updated_idea[:60]}...")
    print(f"  📦 출력 형식 : {state.get('output_format', '미정')}")
    print(f"  🔧 활성 모듈 : {state.get('active_modules', [])}")

    return {
        **state,
        "idea":               updated_idea,
        "user_feedback":      "",     # 통합 완료 후 클리어
        # 재진입 시 카운터 초기화 (새 라운드 시작)
        "level1_retry_count": 0,
        "level2_retry_count": 0,
        "level1_prev_output": "",     # [수정 2] 수렴 비교 초기화
        "convergence_reached": False,
        "session_log": log(state, f"인풋 확인: {updated_idea[:40]}..."),
    }


# ── [수정 1] 노드 3: 에이전트 세트 구성 (신규 노드) ──────────
def node_crew_setup(state: AgentState) -> AgentState:
    """
    다이어그램 ③ 에이전트 세트 구성 노드.
    active_modules 플래그를 읽어 crew_config를 빌드하고 State에 저장한다.
    이 crew_config는 Level 1·2 노드에서 사용해 에이전트를 동적으로 생성한다.

    crew_config 구조:
      {
        "core": ["strategist", "executor", "critic"],
        "modules": {
          "designer": {"role": "UI/UX 디자이너", "goal": "사용자 경험과 시각적 디자인 최적화"},
          ...
        }
      }
    """
    print("\n" + "="*60)
    print("③ 에이전트 세트 구성 (crew_setup)")
    print("="*60)

    # 사용 가능한 모듈 레지스트리
    MODULE_REGISTRY: Dict[str, Dict[str, str]] = {
        "designer": {
            "role": "UI/UX 디자이너",
            "goal": "사용자 경험과 시각적 디자인을 최적화해 전환율을 높인다",
        },
        "marketer": {
            "role": "마케터",
            "goal": "시장 포지셔닝과 성장 전략을 수립해 초기 사용자를 확보한다",
        },
        "bm": {
            "role": "BM(비즈니스 모델) 전문가",
            "goal": "지속 가능한 수익 모델을 설계하고 유닛 이코노믹스를 검증한다",
        },
        # 추가 모듈은 여기에 등록
    }

    active = state.get("active_modules", [])
    selected_modules = {
        name: info
        for name, info in MODULE_REGISTRY.items()
        if name in active
    }

    crew_config: Dict[str, Any] = {
        "core":    ["strategist", "executor", "critic"],
        "modules": selected_modules,
    }

    print(f"  코어 에이전트 : {crew_config['core']}")
    print(f"  활성 모듈     : {list(selected_modules.keys()) or ['없음']}")

    return {
        **state,
        "crew_config": crew_config,
        "session_log": log(state, f"크루 구성: 코어 3 + 모듈 {len(selected_modules)}개"),
    }


# ── 노드 4: Level 1 실행 ───────────────────────────────────
def node_level1(state: AgentState) -> AgentState:
    """
    Level 1: CrewAI 코어 크루 + 활성 모듈 협업.
    Gemma(저비용 LLM) 담당 — Claude 개입 없음.

    [수정 2] 수렴 감지:
      현재 출력과 이전 출력의 Jaccard 유사도를 계산해
      CONVERGENCE_SIMILARITY(0.8) 이상이면 수렴 플래그 설정.
      수렴 또는 MAX_RETRIES 도달 시 CP1으로 전달.

    [수정 7] 카운터 표시:
      시도 횟수 분모를 MAX_RETRIES로 정정 (v1은 MAX_RETRIES+1 오류).
    """
    retry      = state.get("level1_retry_count", 0)
    crew_config = state.get("crew_config", {"core": [], "modules": {}})

    print("\n" + "="*60)
    # [수정 7] 분모 MAX_RETRIES (v1에서 MAX_RETRIES+1 오류 수정)
    print(f"④ Level 1 — CrewAI 협업 (시도 {retry + 1}/{MAX_RETRIES})")
    print("="*60)

    combined_rules = state["master_rules"] + "\n" + state["project_rules"]

    if MOCK_MODE:
        print("  [MOCK] 에이전트 협업 시뮬레이션...")
        # 수렴 여부: 이전 비판가가 "추가 맹점 없음"을 말했다면 수렴 중
        converging = (retry >= 2)  # 3번째 시도부터 수렴 시뮬레이션

        strategist_out = mock_strategist_response(state["idea"], retry)
        executor_out   = mock_executor_response(state["idea"], retry)
        critic_out     = mock_critic_response(retry, converging)

        # 선택 모듈 Mock 출력
        modules_out: Dict[str, str] = {}
        for mod_name, mod_info in crew_config.get("modules", {}).items():
            modules_out[mod_name] = (
                f"[{mod_info['role']} — Mock]\n"
                f"핵심 제안 3가지:\n"
                f"1. {mod_info['role']} 관점에서 아이디어 분석 완료\n"
                f"2. 구체적 액션 아이템 수립됨\n"
                f"3. 다음 단계 로드맵 제시"
            )
    else:
        # ── 실제 CrewAI 실행 ──────────────────────────────────
        crew   = create_core_crew(state["idea"], combined_rules, crew_config, retry)
        result = crew.kickoff()
        outputs = result.tasks_output

        strategist_out = str(outputs[0].raw) if len(outputs) > 0 else "출력 없음"
        executor_out   = str(outputs[1].raw) if len(outputs) > 1 else "출력 없음"
        critic_out     = str(outputs[2].raw) if len(outputs) > 2 else "출력 없음"

        # [수정 5] 모듈 에이전트 출력 추출 (코어 태스크 3개 이후)
        modules_out = {}
        module_names = list(crew_config.get("modules", {}).keys())
        for i, mod_name in enumerate(module_names):
            task_idx = 3 + i
            if task_idx < len(outputs):
                modules_out[mod_name] = str(outputs[task_idx].raw)

    # [수정 2] 수렴 감지: 이전 출력과 현재 출력 Jaccard 유사도 비교
    current_combined = " ".join([strategist_out, executor_out, critic_out,
                                  " ".join(modules_out.values())])
    prev_combined    = state.get("level1_prev_output", "")
    similarity       = jaccard_similarity(prev_combined, current_combined)
    convergence_reached = similarity >= CONVERGENCE_SIMILARITY and retry > 0

    print(f"\n  수렴 유사도: {similarity:.2%}  |  수렴: {'✅' if convergence_reached else '⏳'}")
    print(f"  [전략가 요약] {strategist_out[:120].strip()}...")
    print(f"  [비판가 요약] {critic_out[:120].strip()}...")

    return {
        **state,
        "level1_strategist":  strategist_out,
        "level1_executor":    executor_out,
        "level1_critic":      critic_out,
        "level1_modules":     modules_out,
        "level1_prev_output": current_combined,   # [수정 2] 다음 라운드 비교용 저장
        "convergence_reached": convergence_reached,
        "session_log": log(state, f"Level 1 완료 (시도 {retry+1}, 유사도 {similarity:.1%})"),
    }


# ── 노드 5: 체크포인트 1 ───────────────────────────────────
def node_checkpoint1(state: AgentState) -> AgentState:
    """
    CP1: Claude가 Level 1 결과를 감독한다.
    미통과 → route_after_cp1에서 Level 1 재진입 결정.
    통과   → Level 2 진행.

    [수정 3] Mock 모드에서 첫 시도 fail → 재진입 경로 실제 테스트.
    """
    retry = state.get("level1_retry_count", 0)
    print("\n" + "="*60)
    print("⑤ 체크포인트 1 — Claude 감독")
    print("="*60)

    if MOCK_MODE:
        result, feedback = mock_claude_cp_review(1, retry)
    else:
        try:
            from langchain_anthropic import ChatAnthropic
            llm = ChatAnthropic(model="claude-sonnet-4-20250514")

            prompt = f"""
당신은 멀티에이전트 시스템의 감독관입니다.
아래 Level 1 결과를 검토하고 통과 여부를 판단하세요.

[전역 규칙]
{state['master_rules']}

[프로젝트 규칙]
{state['project_rules']}

[전략가 결과]
{state['level1_strategist']}

[실행가 결과]
{state['level1_executor']}

[비판가 결과]
{state['level1_critic']}

[수렴 도달 여부]
{state.get('convergence_reached', False)}

판단 기준:
1. 완결성: 전략·실행·비판 세 관점 충실한가?
2. 일관성: 에이전트 간 모순이 없는가?
3. 규칙 준수: 전역·프로젝트 규칙을 따랐는가?
4. 비판가 지적: 이전 지적 사항이 반영되었는가?

반드시 이 형식으로 응답하세요:
결과: pass 또는 fail
피드백: (구체적 이유 및 개선 필요 사항)
"""
            response = llm.invoke([HumanMessage(content=prompt)])
            text     = response.content
            result   = "pass" if "결과: pass" in text.lower() else "fail"
            feedback = text

        except Exception as e:
            print(f"  ⚠️  Claude API 오류: {e} → Mock 응답 사용")
            result, feedback = mock_claude_cp_review(1, retry)

    print(f"  CP1 결과 : {'✅ PASS' if result == 'pass' else '❌ FAIL'}")
    print(f"  피드백   : {feedback[:150]}...")

    return {
        **state,
        "cp1_result":         result,
        "cp1_feedback":       feedback,
        "level1_retry_count": retry + 1,   # 재진입 시 사용할 카운터 증가
        "session_log":        log(state, f"CP1 결과: {result} (시도 {retry+1}회)"),
    }


# ── 노드 6: Level 2 실행 ───────────────────────────────────
def node_level2(state: AgentState) -> AgentState:
    """
    Level 2: 에이전트 간 크로스 피드백.
    [수정 5] crew_config를 create_cross_feedback_crew에 전달해
             모듈 에이전트 출력도 통합 조율에 포함.
    """
    retry       = state.get("level2_retry_count", 0)
    crew_config = state.get("crew_config", {"core": [], "modules": {}})

    print("\n" + "="*60)
    print(f"⑥ Level 2 — 크로스 피드백 (시도 {retry + 1}/{MAX_RETRIES})")
    print("="*60)

    l1_results = {
        "strategist": state.get("level1_strategist", ""),
        "executor":   state.get("level1_executor", ""),
        "critic":     state.get("level1_critic", ""),
        "modules":    state.get("level1_modules", {}),
    }
    combined_rules = state["master_rules"] + "\n" + state["project_rules"]

    if MOCK_MODE:
        print("  [MOCK] 크로스 피드백 통합 시뮬레이션...")
        integrated, conflicts = mock_cross_feedback(l1_results, retry)
    else:
        # [수정 5] crew_config 전달
        crew   = create_cross_feedback_crew(l1_results, combined_rules, crew_config)
        result = crew.kickoff()
        integrated = str(result.tasks_output[0].raw) if result.tasks_output else "출력 없음"
        conflicts  = "크루 내부에서 자동 처리됨"

    print(f"  [통합 결과] {integrated[:200].strip()}...")

    return {
        **state,
        "level2_integrated": integrated,
        "level2_conflicts":  conflicts,
        "session_log":       log(state, f"Level 2 완료 (시도 {retry+1})"),
    }


# ── 노드 7: 체크포인트 2 ───────────────────────────────────
def node_checkpoint2(state: AgentState) -> AgentState:
    """
    CP2: Claude가 Level 2 크로스 피드백 결과를 감독한다.
    미통과 → Level 2 재진입.  통과 → CP3 진행.

    [수정 3] Mock: retry=0 fail, retry>=1 pass (CP1과 동일 패턴).
    """
    retry = state.get("level2_retry_count", 0)
    print("\n" + "="*60)
    print("⑦ 체크포인트 2 — Claude 감독")
    print("="*60)

    if MOCK_MODE:
        result, feedback = mock_claude_cp_review(2, retry)
    else:
        try:
            from langchain_anthropic import ChatAnthropic
            llm = ChatAnthropic(model="claude-sonnet-4-20250514")

            prompt = f"""
Level 2 크로스 피드백 결과를 검토하세요.

[전역 규칙]
{state['master_rules']}

[감지된 충돌 목록]
{state.get('level2_conflicts', '없음')}

[통합 결과]
{state.get('level2_integrated', '없음')}

판단 기준:
1. 충돌이 명시적으로 해결되었는가?
2. 통합안이 내부적으로 일관성 있는가?
3. 비판가 지적 사항이 최종 통합안에 반영되었는가?
4. 전역 규칙 준수 여부

반드시 이 형식으로 응답하세요:
결과: pass 또는 fail
피드백: (구체적 사유)
"""
            response = llm.invoke([HumanMessage(content=prompt)])
            text     = response.content
            result   = "pass" if "결과: pass" in text.lower() else "fail"
            feedback = text

        except Exception as e:
            print(f"  ⚠️  Claude API 오류: {e} → Mock 응답 사용")
            result, feedback = mock_claude_cp_review(2, retry)

    print(f"  CP2 결과 : {'✅ PASS' if result == 'pass' else '❌ FAIL'}")

    return {
        **state,
        "cp2_result":         result,
        "cp2_feedback":       feedback,
        "level2_retry_count": retry + 1,
        "session_log":        log(state, f"CP2 결과: {result} (시도 {retry+1}회)"),
    }


# ── 노드 8: 체크포인트 3 (사용자 최종 승인) ───────────────
def node_checkpoint3(state: AgentState) -> AgentState:
    """
    CP3: 사용자 최종 승인 게이트 (Human-in-the-Loop).
    Claude가 제안서 정리 → 사용자 입력 대기.

    프로덕션 전환 시:
      graph.compile(interrupt_before=["checkpoint3"])
      app.update_state(config, {"user_decision": "approved"})
    로 대체해 Webhook / Slack / UI 버튼과 연동 가능.

    [수정 4] 거절 시 user_feedback 수집 → node_process_input에서 idea에 통합.
    """
    print("\n" + "="*60)
    print("⑧ 체크포인트 3 — 사용자 최종 승인 게이트")
    print("="*60)

    # 최종 제안서 생성
    proposal = _generate_proposal(state)
    print(proposal)

    # 사용자 입력 대기
    while True:
        ans = input(
            "\n🙋 위 제안서를 승인하시겠습니까? "
            "[y=승인 / n=거절·피드백입력 / q=세션종료]: "
        ).strip().lower()
        if ans in ("y", "n", "q"):
            break
        print("  y, n, q 중 하나를 입력하세요.")

    if ans == "q":
        decision, feedback = "rejected", "사용자가 세션을 종료했습니다."
    elif ans == "y":
        decision, feedback = "approved", "사용자 승인 완료"
    else:
        # [수정 4] 거절 피드백 수집 → node_process_input에서 idea에 통합
        feedback = input(
            "  💬 어떤 부분을 수정할까요?\n  (이 내용이 다음 라운드 아이디어에 통합됩니다): "
        ).strip()
        decision = "rejected"

    print(f"\n  결정: {'✅ 승인' if decision == 'approved' else '❌ 거절'}")

    return {
        **state,
        "final_proposal": proposal,
        "user_decision":  decision,
        "user_feedback":  feedback,  # node_process_input에서 idea에 통합됨
        "session_log":    log(state, f"사용자 결정: {decision}"),
    }


def _generate_proposal(state: AgentState) -> str:
    """Claude API 또는 Mock으로 최종 제안서 생성"""
    if MOCK_MODE:
        return mock_final_proposal(state)
    try:
        from langchain_anthropic import ChatAnthropic
        llm = ChatAnthropic(model="claude-sonnet-4-20250514")
        prompt = f"""
아래 멀티에이전트 협업 결과를 사용자에게 제출할 최종 제안서로 정리하세요.
출력 형식: {state.get('output_format', '웹앱')}

[전략가 결과]
{state.get('level1_strategist', '')}

[실행가 결과]
{state.get('level1_executor', '')}

[비판가 최종 검토]
{state.get('level1_critic', '')}

[Level 2 크로스 피드백 통합]
{state.get('level2_integrated', '')}

사용자가 한눈에 이해할 수 있도록 명확하고 간결하게, 승인/거절 판단이
쉽도록 핵심 위험·기회·실행 계획을 요약하세요.
"""
        return llm.invoke([HumanMessage(content=prompt)]).content
    except Exception as e:
        print(f"  ⚠️  Claude API 오류: {e} → Mock 제안서 사용")
        return mock_final_proposal(state)


# ── 노드 9: 실행 ────────────────────────────────────────────
def node_execute(state: AgentState) -> AgentState:
    """
    실행 노드: 승인된 안을 실제로 반영하고 세션을 아카이브한다.
    실제 환경에서는 GitHub 커밋, Notion 업로드, Slack 공유 등 수행.
    """
    print("\n" + "="*60)
    print("⑨ 실행 — 승인된 안 반영")
    print("="*60)

    result = (
        f"[실행 완료]\n"
        f"프로젝트 : {state['project_name']}\n"
        f"처리 내용: {state.get('output_format', '결과물')} 생성 완료\n"
        f"저장 위치: ./output/{state['project_name'].replace(' ', '_')}/\n"
        f"아카이브 : ./archive/season_1/ 에 세션 로그 저장\n"
        f"다음 단계: 시즌 2 대기 중"
    )
    print(result)

    # 세션 로그 파일 저장
    log_file = f"{state['project_name'].replace(' ', '_')}_session.txt"
    try:
        with open(log_file, "w", encoding="utf-8") as f:
            f.write(f"프로젝트: {state['project_name']}\n")
            f.write(f"아이디어: {state['idea']}\n\n")
            f.write("=== 세션 로그 ===\n")
            for entry in state.get("session_log", []):
                f.write(entry + "\n")
            f.write("\n=== 최종 제안서 ===\n")
            f.write(state.get("final_proposal", ""))
        print(f"\n  📄 세션 로그: {log_file}")
    except Exception as e:
        print(f"  ⚠️  로그 저장 실패: {e}")

    return {
        **state,
        "execution_result": result,
        "session_log": log(state, "실행 완료 · 아카이브 저장"),
    }




In [ ]:
# ===========================================================
#  ⬇ CELL 9 — 조건부 엣지 라우터 (분기 로직)
# ===========================================================

def route_after_cp1(state: AgentState) -> str:
    """
    CP1 결과 + 수렴 여부에 따라 다음 노드 결정.

    pass OR 수렴 도달        → Level 2 진행
    fail + 재시도 가능       → Level 1 재진입
    fail + MAX_RETRIES 초과  → Level 2 강제 진행 (무한 루프 방지)

    주의: level1_retry_count는 node_checkpoint1에서 이미 +1 된 상태.
          따라서 retry < MAX_RETRIES 조건이 올바르게 동작함.
    """
    result      = state.get("cp1_result", "fail")
    retry       = state.get("level1_retry_count", 0)   # 이미 증가된 값
    converged   = state.get("convergence_reached", False)

    if result == "pass" or converged:
        print("  ➡️  CP1 통과 (또는 수렴) → Level 2 진행")
        return "level2"
    elif retry < MAX_RETRIES:
        print(f"  🔄 CP1 미통과 → Level 1 재진입 ({retry}/{MAX_RETRIES})")
        return "level1"
    else:
        print(f"  ⚠️  최대 재시도({MAX_RETRIES}) 초과 → Level 2 강제 진행")
        return "level2"


def route_after_cp2(state: AgentState) -> str:
    """
    CP2 결과에 따라 다음 노드 결정.
    pass              → CP3 진행
    fail + 재시도 가능 → Level 2 재진입
    fail + 한계 초과   → CP3 강제 진행
    """
    result  = state.get("cp2_result", "fail")
    retry   = state.get("level2_retry_count", 0)

    if result == "pass":
        print("  ➡️  CP2 통과 → 사용자 승인 게이트 진행")
        return "checkpoint3"
    elif retry < MAX_RETRIES:
        print(f"  🔄 CP2 미통과 → Level 2 재진입 ({retry}/{MAX_RETRIES})")
        return "level2"
    else:
        print(f"  ⚠️  최대 재시도({MAX_RETRIES}) 초과 → CP3 강제 진행")
        return "checkpoint3"


def route_after_cp3(state: AgentState) -> str:
    """
    사용자 결정에 따라 다음 노드 결정.
    approved → 실행 노드
    rejected → 인풋 노드 재진입 (user_feedback → idea 통합은 node_process_input에서 처리)
    """
    decision = state.get("user_decision", "rejected")
    if decision == "approved":
        print("  ✅ 사용자 승인 → 실행 시작")
        return "execute"
    else:
        print("  🔄 사용자 거절 → 인풋 재진입 (피드백 통합)")
        return "input"




In [ ]:
# ===========================================================
#  ⬇ CELL 10 — LangGraph 그래프 구성 및 컴파일
#  [수정 1] crew_setup 노드 추가 (input → crew_setup → level1)
# ===========================================================

def build_graph() -> StateGraph:
    """
    전체 워크플로우를 LangGraph StateGraph로 구성.

    [수정 1] crew_setup 노드 추가:
      구.  flow: input → level1
      신.  flow: input → crew_setup → level1

    노드(Node)      : 각 처리 단계 함수
    엣지(Edge)      : 노드 간 고정 연결
    조건부 엣지     : 상태에 따라 동적으로 다음 노드 결정
    """
    graph = StateGraph(AgentState)

    # ── 노드 등록 ─────────────────────────────────────────────
    graph.add_node("rules",        node_load_rules)      # ① 규칙 로드
    graph.add_node("input",        node_process_input)   # ② 인풋 처리
    graph.add_node("crew_setup",   node_crew_setup)      # ③ [수정 1] 에이전트 세트 구성 (신규)
    graph.add_node("level1",       node_level1)          # ④ Level 1 CrewAI
    graph.add_node("checkpoint1",  node_checkpoint1)     # ⑤ CP1 Claude 감독
    graph.add_node("level2",       node_level2)          # ⑥ Level 2 CrewAI
    graph.add_node("checkpoint2",  node_checkpoint2)     # ⑦ CP2 Claude 감독
    graph.add_node("checkpoint3",  node_checkpoint3)     # ⑧ 사용자 승인
    graph.add_node("execute",      node_execute)         # ⑨ 실행

    # ── 고정 엣지 ──────────────────────────────────────────────
    graph.add_edge(START,         "rules")
    graph.add_edge("rules",       "input")
    graph.add_edge("input",       "crew_setup")    # [수정 1] input → crew_setup
    graph.add_edge("crew_setup",  "level1")        # [수정 1] crew_setup → level1
    graph.add_edge("level1",      "checkpoint1")
    graph.add_edge("level2",      "checkpoint2")
    graph.add_edge("execute",     END)

    # ── 조건부 엣지 ────────────────────────────────────────────
    graph.add_conditional_edges(
        "checkpoint1",
        route_after_cp1,
        {"level1": "level1", "level2": "level2"},
    )
    graph.add_conditional_edges(
        "checkpoint2",
        route_after_cp2,
        {"level2": "level2", "checkpoint3": "checkpoint3"},
    )
    graph.add_conditional_edges(
        "checkpoint3",
        route_after_cp3,
        {"execute": "execute", "input": "input"},  # 거절 → input → crew_setup → level1
    )

    return graph




In [ ]:
# ===========================================================
#  ⬇ CELL 11 — 메인 실행 함수
# ===========================================================

def run_season(
    idea:           str,
    project_name:   str  = "나의 프로젝트",
    output_format:  str  = "웹앱",
    constraints:    str  = "1인 개발, 3개월 이내",
    active_modules: list = None,
) -> Dict[str, Any]:
    """
    멀티에이전트 시즌을 실행한다.

    Args:
        idea           : 사용자 아이디어 (자유 텍스트)
        project_name   : 프로젝트 이름
        output_format  : 원하는 결과물 형식
        constraints    : 핵심 제약 조건
        active_modules : 활성 선택 모듈 ["designer", "marketer", "bm"]

    Returns:
        최종 AgentState 딕셔너리
    """
    if active_modules is None:
        active_modules = []

    print("\n" + "🚀 " * 20)
    print(f"  시즌제 멀티에이전트 시스템 v2 시작")
    print(f"  Mode: {'🎭 MOCK (흐름 테스트)' if MOCK_MODE else '🔥 LIVE (실제 API)'}")
    print("🚀 " * 20)

    # ── 초기 상태 ─────────────────────────────────────────────
    initial_state: AgentState = {
        "master_rules":   "", "project_rules":   "",
        "idea":           idea,
        "project_name":   project_name,
        "output_format":  output_format,
        "constraints":    constraints,
        "active_modules": active_modules,
        # [수정 1] crew_config 초기화
        "crew_config": {"core": [], "modules": {}},
        # Level 1
        "level1_strategist":  "", "level1_executor":    "",
        "level1_critic":      "", "level1_modules":     {},
        "level1_retry_count": 0,
        # [수정 2/6] 수렴 감지 필드
        "level1_prev_output":  "", "convergence_reached": False,
        # Level 2
        "level2_integrated":  "", "level2_conflicts":   "",
        "level2_retry_count": 0,
        # 체크포인트
        "cp1_result": "", "cp1_feedback": "",
        "cp2_result": "", "cp2_feedback": "",
        # CP3 & 실행
        "final_proposal": "", "user_decision": "",
        "user_feedback":  "", "execution_result": "",
        "session_log":    [],
    }

    # ── 그래프 빌드 & 컴파일 ──────────────────────────────────
    graph  = build_graph()
    memory = MemorySaver()   # 동일 thread_id로 재실행 시 이전 상태에서 재개
    app    = graph.compile(checkpointer=memory)

    config = {"configurable": {"thread_id": f"season-{project_name}"}}
    final  = app.invoke(initial_state, config=config)

    # ── 실행 요약 ─────────────────────────────────────────────
    print("\n\n" + "="*60)
    print("📊 세션 요약")
    print("="*60)
    print(f"  프로젝트       : {final['project_name']}")
    print(f"  CP1 결과       : {final.get('cp1_result', '-')}")
    print(f"  CP2 결과       : {final.get('cp2_result', '-')}")
    print(f"  사용자 결정     : {final.get('user_decision', '-')}")
    print(f"  L1 재시도 횟수  : {final.get('level1_retry_count', 0)}회")
    print(f"  L2 재시도 횟수  : {final.get('level2_retry_count', 0)}회")
    print(f"  수렴 도달 여부  : {final.get('convergence_reached', False)}")
    print(f"  세션 로그 항목  : {len(final.get('session_log', []))}개")

    return final




In [ ]:
# ===========================================================
#  ⬇ CELL 12 — 그래프 시각화 (선택)
# ===========================================================

def visualize_graph():
    """LangGraph 그래프를 Mermaid PNG로 렌더링한다."""
    try:
        from IPython.display import display, Image
        graph = build_graph()
        app   = graph.compile()
        display(Image(app.get_graph().draw_mermaid_png()))
        print("✅ 그래프 시각화 완료")
    except Exception as e:
        print(f"시각화 실패: {e}")
        print("  → !pip install pygraphviz 설치 후 재시도")




In [ ]:
# ===========================================================
#  ⬇ CELL 13 — 실행 진입점
# ===========================================================

if __name__ == "__main__":
    # 원하는 프로젝트로 수정하세요
    result = run_season(
        idea="""
        AI 기반 개인 영어 학습 앱.
        사용자의 실력을 분석하고 취약점을 자동으로 찾아서
        매일 맞춤형 20분 커리큘럼을 생성해 준다.
        """,
        project_name   = "AI 영어 코치",
        output_format  = "모바일 앱 (iOS/Android) + 웹 대시보드",
        constraints    = "1인 개발, 예산 200만원, 3개월 MVP 출시",
        active_modules = ["designer", "marketer"],
    )

    print("\n\n" + "="*60)
    print("📝 최종 제안서")
    print("="*60)
    print(result.get("final_proposal", "제안서 없음"))


# ===========================================================
#  💡 FAQ — 자주 묻는 질문
# ===========================================================
#
#  Q: Gemma로 Level 1·2 비용 절감하려면?
#  A: build_llm() 함수에서 변경:
#       Groq (무료 API):
#         return LLM(model="groq/gemma2-9b-it",
#                    api_key=os.environ["GROQ_API_KEY"])
#       Ollama (로컬):
#         return LLM(model="ollama/gemma2:9b",
#                    base_url="http://localhost:11434")
#
#  Q: CP3를 UI 버튼/Webhook으로 연동하려면?
#  A: graph.compile(interrupt_before=["checkpoint3"])
#     실행 중단 후 app.update_state(config, {"user_decision": "approved"})
#     로 재개. LangGraph의 interrupt() 기능 참고.
#
#  Q: 새 선택 모듈을 추가하려면?
#  A: node_crew_setup()의 MODULE_REGISTRY에 새 항목 추가:
#     "security": {"role": "보안 전문가", "goal": "보안 취약점 사전 발굴"}
#     active_modules=["security"]로 활성화.
#
#  Q: 수렴 감지 민감도를 조정하려면?
#  A: CONVERGENCE_SIMILARITY 값을 변경:
#     0.7 = 더 빠른 수렴 (루프 적음)
#     0.9 = 더 엄격한 수렴 (루프 많음)
#
#  Q: v1 대비 핵심 변경 사항 요약?
#  A: 1. crew_setup 노드 추가 (다이어그램 ③ 코드 동기화)
#     2. Jaccard 수렴 감지 (임계값 0.8)
#     3. Mock CP 첫 시도 fail (재진입 루프 테스트 가능)
#     4. CP3 거절 피드백 → idea 자동 통합
#     5. active_modules 에이전트 동적 생성
#     6. AgentState: crew_config, level1_prev_output, convergence_reached 추가
#     7. 카운터 표시 분모 MAX_RETRIES로 정정
#
# ===========================================================
